[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q3/03_comparison.ipynb)

# R2-Q3 Week 4 — Compare gap reduction across the three conditions

### This notebook produces `comparison_summary.json`, the headline finding for your Week 4 paper and final presentation.

This notebook is the head-to-head. It takes the three sets of metrics from Weeks 2 and 3, builds the three-way gap-reduction table (no-aug vs kitchen-sink vs targeted), and applies the statistical test you precommitted in Week 1. Per-category breakdowns where the data supports them; overall otherwise. The conclusions should frame augmentation as a problem-reducer, not a problem-solver — per the page's Consideration 5, the gap will not close completely.

By the end of this notebook you will have:

- A three-way gap-reduction table saved as `comparison_table.parquet`, with per-condition PV→PD gap and (where the data supports it) per-category breakdown.
- A `comparison_summary.json` with the headline gap-reduction figure, the precommitted statistical test result, the per-category breakdown, and a one-paragraph interpretation framed per Consideration 5.
- Submitted final presentation and final paper.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R2-Q3 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r2_q3")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Defensive load of `precommit.json` and all three sets of metrics

This is the comparison notebook — the head-to-head that Weeks 2 and 3 were building toward. You trained three classifiers on PlantVillage: a no-augmentation floor, the kitchen-sink baseline, and the targeted set built from your precommitted mapping. Each was scored on PlantVillage and on PlantDoc in the shared five-category disease space, and each carries a PV→PD gap — how much accuracy it loses moving from lab images to field images. This week asks the question the gaps were measured for: does the targeted set reduce that gap more than the kitchen-sink baseline does, and is the difference real or just noise?

This section loads everything the comparison needs before any of it starts: the decisions you locked in NB00, and the three sets of metrics from Weeks 2 and 3. Loading them up front, defensively, means a missing or stale piece stops you here — with a pointer to the notebook that produces it — rather than failing halfway through the comparison.

Three of the six precommit fields drive this notebook:

- `experimental_design` — confirms you're running the three-condition factorial comparison these notebooks are built around.
- `class_space_remapping` — the two class-label-to-category maps (one for PlantVillage, one for PlantDoc). Section 3 re-scores the reloaded classifiers in that shared category space, the same space Weeks 2 and 3 scored in.
- `statistical_comparison` — the test you committed in Week 1, and the verdict rule that goes with it. Section 3 applies exactly this. Committing it before seeing any numbers is what keeps the verdict honest: the bar for "targeted outperforms" was fixed in Week 1, not chosen after the results were in.

In [ ]:
# Read back the decisions you locked in NB00. This notebook consumes three of
# the six precommits; the load is defensive, so a missing or half-written file
# fails here, with a clear message, rather than partway through the comparison.
import json

precommit_path = OUTPUT_DIR / "precommit.json"
if not precommit_path.exists():
    raise FileNotFoundError(
        f"No precommit.json at {precommit_path}. "
        "Run NB00 (00_orient_and_precommit.ipynb) to completion first — "
        "it writes the Week-1 decisions this notebook reads."
    )

with open(precommit_path) as f:
    precommit = json.load(f)

# NB03 consumes three of the six precommit fields:
#   experimental_design    — confirms the three-condition factorial setup.
#   class_space_remapping  — the two class->category maps; Section 3 re-scores
#                            the reloaded classifiers in that shared space.
#   statistical_comparison — the test (and verdict rule) Section 3 applies.
for key in ("experimental_design", "class_space_remapping", "statistical_comparison"):
    if precommit.get(key) is None:
        raise KeyError(
            f"precommit.json is missing '{key}'. Re-run NB00 — every field "
            "should be populated before you start Week 4."
        )

design = precommit["experimental_design"]
csr = precommit["class_space_remapping"]
sc = precommit["statistical_comparison"]

# This notebook compares the three named factorial conditions head to head. A
# hold-one-out design would mean a different comparison, so flag it rather than
# silently running the wrong shape.
if design["design"] != "factorial":
    print(
        f"WARNING: experimental_design is {design['design']!r}, not "
        "'factorial'. This notebook compares the three named factorial "
        "conditions; another design needs a different comparison. Check with "
        "your mentor before continuing."
    )

# Section 3 implements the 'bootstrap_ci' path. The precommit menu also allows
# 'mcnemar' and 'paired_runs'; if you committed one of those, Section 3 stops
# and points you to your mentor, so surface it here too.
if sc["test"] != "bootstrap_ci":
    print(
        f"NOTE: statistical_comparison test is {sc['test']!r}. This notebook "
        "implements the 'bootstrap_ci' path; Section 3 will stop on any other "
        "choice. Decide with your mentor before continuing."
    )

# Print back what drives the rest of the notebook.
print("Loaded precommit.json")
print(f"  Experimental design: {design['design']} "
      f"({design['n_conditions']} conditions)")
print(f"  Class-space remap:   PV {len(csr['pv_class_to_category'])} + "
      f"PD {len(csr['pd_class_to_category'])} classes -> "
      f"{len(csr['categories_used'])} categories")
print(f"  Statistical test:    {sc['test']} on {sc['comparison_target']}")
print(f"  Verdict rule:        {sc['verdict_rule']}")

The comparison reads three sets of numbers — one per condition. NB01 wrote two files and NB02 wrote the third, all to this question's output directory:

- `baseline_metrics.parquet` — the no-augmentation floor.
- `kitchen_sink_metrics.parquet` — the kitchen-sink baseline.
- `targeted_metrics.parquet` — the targeted condition.

Each has the same shape: one row per disease category plus a final `overall` row, with columns `category`, `pv_accuracy`, `pd_accuracy`, and `gap` (PlantVillage accuracy minus PlantDoc accuracy). A cell is blank wherever a category has no examples in PlantDoc's test split — `pest` is the known case — and that's expected, not a problem.

These files carry point estimates: one accuracy per cell, no spread. That is enough for the three-way table in Section 2 and the per-category breakdown in Section 4. It is *not* enough for the statistical test in Section 3, which needs the per-image results underneath these numbers — so Section 3 reloads the classifiers and re-runs them. Loading the metrics now, defensively, means a missing file stops you here rather than after that re-inference.

In [ ]:
# Load all three sets of metrics from Weeks 2 and 3 — the no-augmentation floor,
# the kitchen-sink baseline, and the targeted condition. The whole comparison is
# built from these, so load them defensively: a missing or stale file should
# fail here, naming the notebook that writes it, not mis-compare silently later.
import pandas as pd

METRIC_FILES = {
    "baseline":     "baseline_metrics.parquet",      # NB01, no-augmentation floor
    "kitchen_sink": "kitchen_sink_metrics.parquet",  # NB01, strong baseline
    "targeted":     "targeted_metrics.parquet",      # NB02, the experimental set
}
WROTE_BY = {
    "baseline":     "NB01 (01_baseline_and_kitchen_sink.ipynb)",
    "kitchen_sink": "NB01 (01_baseline_and_kitchen_sink.ipynb)",
    "targeted":     "NB02 (02_targeted_augmentation.ipynb)",
}
EXPECTED_METRIC_COLS = {"category", "pv_accuracy", "pd_accuracy", "gap"}

metrics = {}
for name, fname in METRIC_FILES.items():
    path = OUTPUT_DIR / fname
    if not path.exists():
        raise FileNotFoundError(
            f"No {fname} at {path}. Run {WROTE_BY[name]} to completion first — "
            "it writes the metrics this notebook compares."
        )
    df = pd.read_parquet(path)

    missing = EXPECTED_METRIC_COLS - set(df.columns)
    if missing:
        raise KeyError(
            f"{fname} is missing columns {sorted(missing)}. This usually means "
            f"an older schema — re-run {WROTE_BY[name]} to regenerate it."
        )
    if "overall" not in set(df["category"]):
        raise ValueError(
            f"{fname} has no 'overall' row. The overall PV->PD gap is the "
            f"headline number this notebook compares — re-run {WROTE_BY[name]}."
        )
    metrics[name] = df


def overall_gap(df):
    """The overall PV->PD gap for one condition: its 'overall' row's gap."""
    return float(df.loc[df["category"] == "overall", "gap"].iloc[0])


# Read back the three overall gaps — the numbers the whole comparison turns on.
# Gap reduction (Section 2) is the floor's gap minus each condition's gap; the
# headline is how much more the targeted set reduces it than kitchen-sink does.
print("Loaded three sets of metrics:")
for name in METRIC_FILES:
    print(f"  {name:13s} overall PV->PD gap = {overall_gap(metrics[name]):.3f}")

One configuration cell before the comparison. As in Weeks 2 and 3, `DEV_MODE` chooses the dataset size — the tiny variant while you get the notebook running end to end, the full variant for the numbers you report. This notebook trains nothing, so that is all the knob does here.

There is a coupling to be careful about this week. Section 3 re-runs the two trained classifiers to recover their per-image results, and it does that on the same PlantVillage and PlantDoc test splits that produced the metrics files you just loaded. So `DEV_MODE` here has to match the one you used in Weeks 2 and 3: re-scoring the full splits when the saved metrics came from the tiny variant (or the reverse) would put the test next to numbers that were never measured on the same images. Section 3 checks for exactly this and stops if the numbers don't line up — but matching it now saves you the re-run.

In [ ]:
# Run configuration — set once here; Section 3 reads from it.
#
# DEV_MODE is the one knob. It picks which size of the datasets Section 3
# reloads when it re-runs inference to get the per-image results the bootstrap
# needs:
#   True  -> the "tiny" variant. Re-inference finishes in a minute or two.
#   False -> the "full" variant. The real run.
#
# IMPORTANT: set this to MATCH the DEV_MODE you used in Weeks 2 and 3. Section 3
# re-scores the reloaded classifiers on the SAME test splits that produced the
# metrics files you just loaded, and checks its own numbers against them. If the
# variant here doesn't match, that check fails loud — which is the point, but it
# means a mismatch costs you a re-run.
DEV_MODE = True

DATA_VARIANT = "tiny" if DEV_MODE else "full"

# One seed for the notebook. The bootstrap in Section 3 also takes its own
# seeded generator, but seeding here keeps data loading and evaluation ordering
# reproducible too.
iri.seed_all(42)

# Section 3 reloads two classifiers and runs them, so a GPU is required — same
# as the training notebooks. Fail now, with instructions, rather than partway
# through Section 3.
assert iri.has_gpu(), (
    "No GPU detected. Section 3 re-runs inference with two ResNet-18 "
    "classifiers and needs a GPU. In Colab: Runtime -> Change runtime type -> "
    "T4 GPU, then re-run from the top."
)

print(f"DEV_MODE     = {DEV_MODE}")
print(f"DATA_VARIANT = {DATA_VARIANT!r}  (must match Weeks 2-3)")
print("Seed set to 42; GPU detected.")

### 2) Build the three-way gap-reduction table (no-aug vs kitchen-sink vs targeted)

With the three sets of metrics loaded, the first thing to build is the descriptive picture: where each condition's PV→PD gap sits, and how much each augmentation strategy shrank it relative to doing nothing. This is the table you will report; Section 3 then asks whether the difference between the two augmentation strategies is more than noise.

Two ideas organize the table. The **gap** is what each condition loses moving from PlantVillage (lab) to PlantDoc (field) — its overall PV accuracy minus its overall PD accuracy, already sitting in each metrics file. **Gap reduction** is the comparison this week is built on: how much smaller a condition's gap is than the no-augmentation floor's. The floor establishes how large the lab→field gap is when you do nothing; kitchen-sink and targeted each pull it down by some amount, and the question is which pulls it down more.

### Practice 2.1 — Compute each condition's gap reduction

Gap *reduction* is the week's central quantity: how much smaller a condition's PV→PD gap is than the no-augmentation floor's. For a condition `c`, it is the floor's overall gap minus `c`'s overall gap:

    gap_reduction(c) = gap(baseline) - gap(c)

A larger reduction means that condition's augmentation did more to close the lab→field gap. The no-augmentation floor is the reference, so by this definition its own reduction is zero.

Build a dictionary named `gap_reduction` that maps each of the three condition names (`"baseline"`, `"kitchen_sink"`, `"targeted"`) to its gap reduction. You already have what you need: `overall_gap(metrics[name])` gives any condition's overall gap, and `metrics` is keyed by those three names.

The table cell below reads `gap_reduction`, so if you skip this it will stop with a `NameError`.

In [ ]:
### Practice 2.1 — your code here --------------------------------------------------
#
#
#
#
#
# -----------------------------------------------------------------------------------

In [ ]:
# Assemble the three-way overall comparison from the point estimates you loaded
# in Section 1. One row per condition: its PV-internal accuracy, its PlantDoc
# accuracy, the PV->PD gap, and the gap reduction you just computed. The baseline
# row is the reference, so its reduction is 0 by construction.
import pandas as pd


def overall_value(df, col):
    """One condition's overall-row value for a column (pv_accuracy, etc.)."""
    return float(df.loc[df["category"] == "overall", col].iloc[0])


overall_table = pd.DataFrame([
    {
        "condition":     name,
        "pv_accuracy":   overall_value(metrics[name], "pv_accuracy"),
        "pd_accuracy":   overall_value(metrics[name], "pd_accuracy"),
        "gap":           overall_gap(metrics[name]),
        "gap_reduction": gap_reduction[name],
    }
    for name in METRIC_FILES
])

print(overall_table.to_string(index=False,
                              float_format=lambda v: f"{v:.3f}"))

# The headline number: how much MORE the targeted set reduces the gap than the
# kitchen-sink baseline does. This is the comparison_target you precommitted
# (targeted_minus_kitchen_sink_gap_reduction), as a point estimate. It is
# positive when the targeted gap is smaller than the kitchen-sink gap — i.e.
# when targeting the failure modes beat the broad policy. Section 3 puts a
# confidence interval around this same number and decides whether it is real.
headline_gap_reduction = gap_reduction["targeted"] - gap_reduction["kitchen_sink"]

print()
print(f"  Targeted gap reduction:     {gap_reduction['targeted']:+.3f}")
print(f"  Kitchen-sink gap reduction: {gap_reduction['kitchen_sink']:+.3f}")
print(f"  Targeted - kitchen-sink:    {headline_gap_reduction:+.3f}   "
      "(precommitted comparison target, point estimate)")

Read the table from the two augmentation rows, not the floor. The floor is there to set the scale — it is how large the gap is with no augmentation at all. Beating the floor only shows that augmentation helps, which Week 2 already established. The comparison that carries the week is **targeted versus kitchen-sink**: both are augmentation strategies, and the question is whether *targeting* the failure modes beats throwing a broad policy at the problem. A positive headline number means the targeted set reduced the gap more than kitchen-sink did.

Two cautions before you over-read it. First, this is a point estimate — one number per condition, with no sense of how much it would move if you reran the evaluation on a different sample of test images. A small positive headline could be a real effect or could be noise, and telling those apart is exactly the precommitted bootstrap test in Section 3. Second, keep Consideration 5 in view: even the targeted condition's gap is still well above zero. Augmentation is *reducing* the lab→field gap, not closing it — the structural confound in PlantVillage is too entrenched for that, and the write-up should say so.

### 3) Apply the precommitted statistical test

This is the test the whole question turns on. Section 2 gave you a point estimate — targeted reduced the gap by some amount more (or less) than kitchen-sink — but a single number cannot tell you whether that difference is real or an artifact of which images happened to land in the test splits. The precommitted test answers that: a bootstrap confidence interval on the targeted-minus-kitchen-sink gap reduction, with the verdict fixed back in Week 1 as "the interval excludes zero."

A bootstrap resamples the test images many times, recomputing the difference on each resample, to see how much it moves. That needs the per-image results underneath the accuracies — which images each classifier got right — and the metrics files you loaded carry only the averaged accuracies, not the per-image record. So this section reloads the two augmentation classifiers and re-runs them on the same PlantVillage and PlantDoc test splits, recovering those per-image results.

Before resampling anything, it checks one thing: that the accuracies it just recomputed match the ones in your saved metrics files. They should match exactly — the evaluation is deterministic — so a mismatch means the re-inference ran on different data than Weeks 2 and 3 did, almost always a `DATA_VARIANT` that does not match the `DEV_MODE` you trained under. The check stops you there rather than letting you bootstrap numbers that do not correspond to your reported results.

In [ ]:
# This notebook implements the bootstrap-CI path. The precommit menu also allows
# 'mcnemar' and 'paired_runs'; those need a different Section 3, so stop here
# rather than run the wrong test on your committed choice.
if sc["test"] != "bootstrap_ci":
    raise NotImplementedError(
        f"This notebook implements the 'bootstrap_ci' path; your precommit "
        f"committed {sc['test']!r}. The 'mcnemar' and 'paired_runs' options "
        "need a different Section 3 — work it out with your mentor."
    )

# Reload the two augmentation classifiers NB01 and NB02 saved. The no-aug floor
# is NOT reloaded: it cancels out of the comparison target (its gap appears in
# both gap reductions and subtracts away), so the test needs only kitchen-sink
# and targeted. map_location='cpu' keeps the load device-agnostic;
# evaluate_in_categories moves the rebuilt model to the GPU itself.
import torch

CLASSIFIER_FILES = {
    "kitchen_sink": "kitchen_sink_classifier.pt",  # NB01
    "targeted":     "targeted_classifier.pt",       # NB02
}
WROTE_CLASSIFIER = {
    "kitchen_sink": "NB01 (01_baseline_and_kitchen_sink.ipynb)",
    "targeted":     "NB02 (02_targeted_augmentation.ipynb)",
}

state_dicts = {}
for name, fname in CLASSIFIER_FILES.items():
    path = OUTPUT_DIR / fname
    if not path.exists():
        raise FileNotFoundError(
            f"No {fname} at {path}. Run {WROTE_CLASSIFIER[name]} to completion "
            "first — it saves the trained weights this notebook reloads."
        )
    state_dicts[name] = torch.load(path, map_location="cpu")

print(f"Reloaded {len(state_dicts)} classifiers: {', '.join(state_dicts)}.")

In [ ]:
# Load both datasets at the variant set in Section 1, and build each one's
# class_idx -> category lookup from its committed map. This is the same setup
# Weeks 2 and 3 used to score in category space; the coverage assert inside
# build_idx_to_cat confirms every class is in the committed map.
pv_metadata, pv_hf = iri.load_plantvillage(variant=DATA_VARIANT)
pd_metadata, pd_hf = iri.load_plantdoc(variant=DATA_VARIANT)

num_classes = pv_metadata["class_idx"].nunique()
categories = csr["categories_used"]

pv_idx_to_cat = iri.build_idx_to_cat(
    pv_metadata, csr["pv_class_to_category"], "PlantVillage"
)
pd_idx_to_cat = iri.build_idx_to_cat(
    pd_metadata, csr["pd_class_to_category"], "PlantDoc"
)

pv_test = pv_metadata[pv_metadata["split"] == "test"]
pd_test = pd_metadata[pd_metadata["split"] == "test"]

print(f"PV test split: {len(pv_test)} images; PD test split: {len(pd_test)} images.")

Score both classifiers the same way Weeks 2 and 3 did — in the five-category space, on each test split — but this time ask `evaluate_in_categories` to also hand back the per-image categories it assigned, via `return_per_item=True`. That gives, for each condition and each split, a record of which images landed in the right category and which did not. The predictions always map through PlantVillage's lookup (the model predicts PlantVillage classes no matter what it is shown); the ground truth maps through PlantVillage's lookup for the PV-internal score and PlantDoc's for the PD score — the same asymmetry from Weeks 2 and 3.

Four scorings in all: two classifiers, each on PV-internal and on PD. From each, a per-image correctness vector — `True` where the predicted category matched the true category. Because both conditions are scored on the same test split in the same order, the two vectors for a split line up image-for-image, which is what lets the bootstrap resample image positions and read both conditions at those same positions. The gate assert closes the cell: every recomputed overall accuracy must match the saved metrics, or it stops with a pointer to the likely `DATA_VARIANT` mismatch.

In [ ]:
import numpy as np


def score(name, eval_meta, eval_hf, dataset_class, true_map):
    """Re-score one classifier on one split; return (overall, correctness vector).

    return_per_item gives back the per-image categories evaluate_in_categories
    already computes; correctness is pred-category == true-category per image,
    in eval_meta order. Predictions always map through PlantVillage (the model's
    class space); truth maps through `true_map` (PV for PV-internal, PD for PD).
    """
    res = iri.evaluate_in_categories(
        state_dicts[name], eval_meta, eval_hf, dataset_class,
        num_classes=num_classes,
        pred_idx_to_cat=pv_idx_to_cat,   # model predicts PV classes
        true_idx_to_cat=true_map,
        categories=categories,
        return_per_item=True,
    )
    pi = res["per_item"]
    correct = np.array([p == t for p, t in zip(pi["pred_cats"], pi["true_cats"])])
    return res["overall"], correct


# Two conditions x {PV-internal, PD}. pv_test / pd_test are passed unchanged to
# both conditions, so the correctness vectors for a split align position-for-
# position across conditions.
pv_overall_ks, pv_correct_ks = score("kitchen_sink", pv_test, pv_hf, iri.PlantVillageDataset, pv_idx_to_cat)
pv_overall_tg, pv_correct_tg = score("targeted",     pv_test, pv_hf, iri.PlantVillageDataset, pv_idx_to_cat)
pd_overall_ks, pd_correct_ks = score("kitchen_sink", pd_test, pd_hf, iri.PlantDocDataset,     pd_idx_to_cat)
pd_overall_tg, pd_correct_tg = score("targeted",     pd_test, pd_hf, iri.PlantDocDataset,     pd_idx_to_cat)

# Reproducibility gate: every recomputed overall must match the saved metrics.
# Deterministic eval means exact agreement; a mismatch is almost always a
# DATA_VARIANT that doesn't match the DEV_MODE used in Weeks 2-3. (overall_value
# was defined in Section 2.)
GATE_TOL = 1e-6
gate_checks = [
    ("kitchen_sink", "PV", pv_overall_ks, overall_value(metrics["kitchen_sink"], "pv_accuracy")),
    ("kitchen_sink", "PD", pd_overall_ks, overall_value(metrics["kitchen_sink"], "pd_accuracy")),
    ("targeted",     "PV", pv_overall_tg, overall_value(metrics["targeted"],     "pv_accuracy")),
    ("targeted",     "PD", pd_overall_tg, overall_value(metrics["targeted"],     "pd_accuracy")),
]
for cond, split, recomputed, saved in gate_checks:
    assert abs(recomputed - saved) < GATE_TOL, (
        f"Re-inferred {cond} {split} accuracy ({recomputed:.4f}) does not match "
        f"the saved metrics ({saved:.4f}). The most likely cause is DATA_VARIANT "
        f"= {DATA_VARIANT!r} not matching the DEV_MODE you used in Weeks 2-3. "
        "Set DEV_MODE to match and re-run from the top."
    )

# The observed statistic, straight from the per-image vectors: the same
# targeted-minus-kitchen-sink gap reduction as Section 2's headline, since the
# gate just confirmed these accuracies equal the saved ones.
observed = ((pv_correct_ks.mean() - pv_correct_tg.mean())
            - (pd_correct_ks.mean() - pd_correct_tg.mean()))

print("Reproducibility gate passed: re-inferred accuracies match the saved metrics.")
print(f"Observed targeted - kitchen-sink gap reduction: {observed:+.3f}")

In [ ]:
# The bootstrap settings come straight from your precommit, and the generator is
# seeded so the interval reproduces. These four arrays and two lengths are what
# the resampling loop in the practice below works with.
n_resamples = sc["bootstrap_params"]["n_resamples"]
ci_level = sc["bootstrap_params"]["ci_level"]
boot_rng = np.random.default_rng(42)

n_pv = len(pv_correct_ks)   # == len(pv_correct_tg)
n_pd = len(pd_correct_ks)   # == len(pd_correct_tg)

print(f"Bootstrap: {n_resamples} resamples, {int(ci_level * 100)}% CI.")
print(f"Resampling {n_pv} PV test images and {n_pd} PD test images per replicate.")

### Practice 3.1 — Run the paired bootstrap

This is the resampling that turns the one observed number into an interval. The idea: build many "alternate test sets" by drawing images with replacement, recompute the gap-reduction difference on each, and see how much it moves. The spread of those recomputed numbers is what tells you whether the observed difference is solid or could easily have come out the other way.

For **one** resample:

1. Draw `n_pv` PlantVillage test positions *with replacement* — `boot_rng.integers(0, n_pv, size=n_pv)` gives that many random positions in `0 .. n_pv-1`, repeats allowed. Draw `n_pd` PlantDoc positions the same way.
2. Read **both** conditions at those same drawn positions. On the PV positions, kitchen-sink's accuracy is `pv_correct_ks[positions].mean()` and targeted's is `pv_correct_tg[positions].mean()` (a boolean vector averages to a proportion). Do the same on the PD positions with `pd_correct_ks` and `pd_correct_tg`.
3. The statistic for this resample is `(pv_ks - pv_tg) - (pd_ks - pd_tg)` — the same targeted-minus-kitchen-sink gap reduction, on this resampled test set.

Using the *same* drawn positions for both conditions is the part that matters: it keeps the comparison paired, so the interval measures the difference between the two strategies on shared images, not the noise of two separate image draws. Repeat all of this `n_resamples` times and collect each statistic into a list or array named `boot_stats`.

The verdict cell below reads `boot_stats`, so if you skip this it will stop with a `NameError`.

In [ ]:
### Practice 3.1 — your code here --------------------------------------------------
#
#
#
#
#
#
#
# -----------------------------------------------------------------------------------

In [ ]:
# Turn the resampled statistics into a confidence interval and read the verdict
# exactly as you precommitted it. The interval is the central ci_level of
# boot_stats (the 2.5th and 97.5th percentiles for a 95% CI); the verdict rule
# is whether that interval excludes zero.
boot_stats = np.asarray(boot_stats)
alpha = 1.0 - ci_level
ci_low, ci_high = np.quantile(boot_stats, [alpha / 2, 1.0 - alpha / 2])

# The page predicts a positive effect (targeted reduces the gap more than
# kitchen-sink), so the informative outcome is an interval lying entirely above
# zero. An interval below zero would be the opposite finding; one straddling
# zero is "not distinguishable from noise" by the committed rule.
excludes_zero = bool(ci_low > 0 or ci_high < 0)

test_result = {
    "test": sc["test"],
    "comparison_target": sc["comparison_target"],
    "observed": float(observed),
    "ci_level": ci_level,
    "ci_low": float(ci_low),
    "ci_high": float(ci_high),
    "n_resamples": int(n_resamples),
    "verdict_rule": sc["verdict_rule"],
    "excludes_zero": excludes_zero,
}

print(f"Observed targeted - kitchen-sink gap reduction: {observed:+.3f}")
print(f"{int(ci_level * 100)}% bootstrap CI: [{ci_low:+.3f}, {ci_high:+.3f}]  "
      f"({n_resamples} resamples)")
print(f"Excludes zero: {excludes_zero}")
if excludes_zero and ci_low > 0:
    print("=> Targeted reliably reduces the gap more than kitchen-sink, "
          "by the precommitted rule.")
elif excludes_zero and ci_high < 0:
    print("=> Kitchen-sink reliably reduces the gap more than targeted — "
          "the opposite of the page's prediction.")
else:
    print("=> The interval includes zero: the difference is not distinguishable "
          "from noise by the precommitted rule.")

Whatever the interval says, read it against the page's Prediction rather than as a pass/fail. The page expected targeted to win, but only by a modest margin — and it said plainly why a *narrow* result is informative: if a broad RandAugment policy reduces the gap nearly as much as the targeted set, that is evidence the kitchen-sink approach incidentally picked up the same signal the targeted set was designed to capture. So an interval that sits just above zero supports the targeted framework modestly; an interval that straddles zero says the two strategies are not distinguishable here, which is a real result about how much *targeting* adds over *breadth*, not a null finding to be embarrassed by.

What the interval does **not** speak to is whether the gap is gone. Even a clear win for targeted is a win at *reducing* the lab→field gap, not closing it — that framing is Section 5's job, and Consideration 5 is firm that the structural confound in PlantVillage keeps the gap open. Before that, Section 4 breaks the comparison down by disease category, to see whether the gap reduction is spread across categories or concentrated where the background cue mattered most.

### 4) Per-failure-mode breakdown where the data supports it (overall otherwise — per Consideration 4)

The overall gap reduction in Section 3 is a single pooled number across all five disease categories. This section opens it up: does the targeted set shrink the gap evenly across categories, or mostly in one or two? The three metrics files already hold per-category gaps, so this is a reshape, not new computation — line the three conditions up side by side, one row per category.

"Where the data supports it" is the catch. A per-category gap needs accuracy on both sides — PlantVillage and PlantDoc — and one category, `pest`, has no images in PlantDoc's test split. So `pest` has a PlantVillage accuracy but no PlantDoc accuracy, and therefore no gap, in any condition. It shows up as blank in the table below; that is the "overall otherwise" case, and it is expected, not a problem.

One caution to carry through the whole section: these per-category numbers are descriptive, and noisier than the pooled overall. Pooling the disease classes into five categories is what made the counts large enough to trust; splitting back out by category thins them again, especially on PlantDoc. So the precommitted test stays on the overall number from Section 3 — the per-category view is for seeing *where* the action is, not for a second verdict.

In [ ]:
# Per-category PV->PD gap, one column per condition. The three metrics files
# already carry it; this just lines them up and orders the rows by the committed
# category list (which also drops the 'overall' row — that is Section 2's
# table). pest comes through as NaN: with no PlantDoc test images, no pest gap
# can be formed in any condition.
gap_by_cat = pd.DataFrame({
    name: metrics[name].set_index("category")["gap"] for name in METRIC_FILES
}).reindex(categories)

print("Per-category PV->PD gap (n/a = no PlantDoc examples):")
print(gap_by_cat.to_string(float_format=lambda v: f"{v:.3f}", na_rep="n/a"))

### Practice 4.1 — Per-category gap reduction, targeted vs kitchen-sink

Section 2 computed gap reduction overall; do the same per category, for the comparison that matters. For each category, the targeted-vs-kitchen-sink reduction is the kitchen-sink gap minus the targeted gap — positive where the targeted set left a *smaller* gap than kitchen-sink did, that is, where targeting helped more.

You have `gap_by_cat`, with a `"kitchen_sink"` column and a `"targeted"` column. Subtract the two to get a per-category Series named `per_category_reduction`. You do not need to do anything special about `pest`: it is NaN in both columns, and subtracting leaves it NaN, which is exactly right.

The cell below reads `per_category_reduction`, so if you skip this it will stop with a `NameError`.

In [ ]:
### Practice 4.1 — your code here --------------------------------------------------
#
#
#
#
#
#
#
# -----------------------------------------------------------------------------------

In [ ]:
# Attach the per-category reduction beside the per-condition gaps. category_table
# is the per-category breakdown Section 6 saves alongside the overall table.
category_table = gap_by_cat.assign(targeted_vs_kitchen_sink=per_category_reduction)

print("Per-category breakdown (gap by condition; last column = targeted-vs-"
      "kitchen-sink reduction):")
print(category_table.to_string(float_format=lambda v: f"{v:.3f}", na_rep="n/a"))

# Where did targeting help most? The category with the largest positive
# reduction, ignoring the NaN categories. Descriptive only — see the note below.
ranked = per_category_reduction.dropna().sort_values(ascending=False)
if len(ranked):
    print(f"\nLargest targeted-vs-kitchen-sink reduction: {ranked.index[0]} "
          f"({ranked.iloc[0]:+.3f})")

Two things to take from this table, and one not to.

Take, first, the shape of the reduction: whether the targeted set's advantage over kitchen-sink is spread across categories or sits in one or two. A reduction concentrated in a single category is a narrower, more specific claim than one spread evenly, and the paper should say which it is.

Take, second, the sign per category. A negative entry means the targeted set left a *larger* gap than kitchen-sink in that category — a place where targeting did not help, or even hurt. That is honest signal, not something to hide; the kitchen-sink policy is broad, and there will be categories where breadth happened to do better.

Do **not** read this table as attributing the reduction to a particular failure mode. The categories here are disease categories (fungal, bacterial, and so on), not the failure modes the targeted set was built against (background, leaf-shape, lighting). Which addressed failure mode plausibly drove a category's reduction is a question for the interpretation in Section 5 — an argument you make from what you know about the images, not a number this table measures. And as the framing cell warned, these per-category figures are descriptive; the verdict stays the pooled bootstrap from Section 3.

### 5) Frame conclusions per Consideration 5 — augmentation reduces the problem, doesn't solve it

Section 3 produced a verdict; this section turns it into the conclusion you will report. The page is specific about how that conclusion should be framed — Consideration 5 — so this is not a free-form summary. The thesis it asks you to hold is: augmentation *reduces* the lab→field gap, it does not close it.

That framing comes from a concrete finding. Noyan (2022) showed PlantVillage's disease labels can be predicted from roughly eight background pixels at about 49% accuracy — the backgrounds are tied to the disease labels, so a model can score well by reading the backdrop instead of the leaf. Background-targeted augmentation is the most direct response to exactly that, which is why the targeted set leans on it. But no augmentation removes the confound; it only makes the shortcut less reliable. So however large your targeted win, the honest claim is about *how much* the gap shrank, never that the problem is solved.

Read your Section 3 verdict against the page's Prediction, not as a pass or fail. The page expected targeted to win by a modest margin, and said why a narrow result is informative: if a broad RandAugment policy (Cubuk et al., 2020) reduces the gap nearly as much as the targeted set, that is evidence kitchen-sink incidentally captured the same signal the targeted set was designed for — a small advantage means targeting succeeded for reasons breadth already covered. A large, clear advantage would instead support the targeted-augmentation framework (Mikołajczyk-Bareła et al., 2023). And an interval that straddles zero is a genuine result too: it says that here, targeting and breadth are not distinguishable — a finding about what targeting adds, not a failure to report.

In [ ]:
# Everything you need in one place to write the interpretation. Nothing new is
# computed here — these are the numbers Sections 2 through 4 already produced.
print("Gap reduction vs the no-augmentation floor:")
for name in ("kitchen_sink", "targeted"):
    print(f"  {name:13s} {gap_reduction[name]:+.3f}")

print("\nTargeted - kitchen-sink (the comparison target):")
print(f"  observed:       {test_result['observed']:+.3f}")
print(f"  {int(test_result['ci_level'] * 100)}% CI:        "
      f"[{test_result['ci_low']:+.3f}, {test_result['ci_high']:+.3f}]")
print(f"  excludes zero:  {test_result['excludes_zero']}")

print("\nPer-category targeted-vs-kitchen-sink reduction:")
for cat, val in per_category_reduction.items():
    shown = f"{val:+.3f}" if pd.notna(val) else "n/a (no PlantDoc examples)"
    print(f"  {cat:10s} {shown}")

Write the interpretation in the cell below — one tight paragraph. Cover, at a minimum:

- **The verdict, in plain terms.** Does the confidence interval exclude zero, and what does that say about targeted versus kitchen-sink? State it as a finding, not a number dump.
- **The magnitude, not just the direction.** Give the observed gap-reduction difference and its interval. "Targeted reduced the gap by X more than kitchen-sink, 95% CI [a, b]" says far more than "targeted won."
- **How to read the margin.** Tie the size of the effect to the page's Prediction — a small-but-real margin as evidence kitchen-sink reached the same fix incidentally; a straddling-zero interval as targeting and breadth being indistinguishable here.
- **The per-category shape.** From Section 4: is the advantage spread across categories or concentrated in one or two, and are there categories where targeting did worse? If you argue a particular failure mode drove it, say so as an interpretation grounded in the images — not as something the category table measured.
- **The Consideration-5 close.** Whatever the verdict, the gap is reduced, not closed. Name the structural reason (backgrounds tied to disease labels; the Noyan 2022 finding) and state the scope honestly: this work measures how much augmentation helps, it does not claim the shortcut is gone.

In [ ]:
# Your one-paragraph interpretation, framed per Consideration 5. Replace the
# placeholder below with your own writing — cover the points in the prompt above
# (the verdict and its magnitude, how to read the margin, the per-category
# shape, and the firm Consideration-5 close that the gap is reduced, not closed).
# Section 6 saves this into comparison_summary.json and will not save a placeholder.
interpretation = (
    "PLACEHOLDER — REWRITE BEFORE SUBMISSION."
)

if interpretation.strip().startswith("PLACEHOLDER"):
    print("NOTE: interpretation is still the placeholder. Rewrite it before the "
          "closeout — Section 6 will refuse to save a placeholder.")
else:
    print(f"Interpretation set ({len(interpretation.split())} words).")

This paragraph is the heart of your Week-4 deliverable. The closeout saves it, alongside the numbers, into `comparison_summary.json`, and it is the conclusion your final paper and presentation build on. Section 6 checks that you have actually written it — a placeholder will stop the save — so finish it here before moving on.

### 6) Closeout: save `comparison_table.parquet` and `comparison_summary.json`; submit final presentation and final paper

Week 4 ends with two files written to your question's output directory. They are the comparison itself, in a form your paper and presentation read from.

- `comparison_table.parquet` — the measurements: every condition's PlantVillage accuracy, PlantDoc accuracy, and PV→PD gap, both overall and per disease category, in one tidy table. The overall rows also carry each condition's gap reduction against the no-augmentation floor.
- `comparison_summary.json` — the findings: the headline targeted-minus-kitchen-sink figure, the full precommitted test result (interval and verdict), the per-category reduction, and your Consideration-5 interpretation from Section 5.

The split is deliberate: the parquet holds the numbers anyone could recompute, the JSON holds what you concluded from them.

In [ ]:
# comparison_table.parquet — the measurements in one tidy table. Stack the three
# metrics files with a `condition` column, so every (condition, category) pair
# has its PV accuracy, PD accuracy, and gap, with the per-category rows and the
# overall row side by side. The overall rows also carry the gap reduction vs the
# floor — a per-condition quantity, so it is left blank on the per-category rows.
comparison_table = pd.concat(
    [metrics[name].assign(condition=name) for name in METRIC_FILES],
    ignore_index=True,
)

comparison_table["gap_reduction"] = np.nan
overall_rows = comparison_table["category"] == "overall"
comparison_table.loc[overall_rows, "gap_reduction"] = (
    comparison_table.loc[overall_rows, "condition"].map(gap_reduction)
)

comparison_table = comparison_table[
    ["condition", "category", "pv_accuracy", "pd_accuracy", "gap", "gap_reduction"]
]

table_path = OUTPUT_DIR / "comparison_table.parquet"
comparison_table.to_parquet(table_path, index=False)
print(f"Wrote {table_path}  ({len(comparison_table)} rows)")

In [ ]:
# comparison_summary.json — the findings. First the gate: the interpretation must
# be written. A placeholder (or empty) interpretation stops the save here, the
# same way NB00 refused a placeholder reasoning — the summary is the conclusion,
# and an unwritten conclusion should not be saved as if it were one.
if interpretation.strip().startswith("PLACEHOLDER") or not interpretation.strip():
    raise RuntimeError(
        "interpretation is still the placeholder (or empty). Write your "
        "Consideration-5 conclusion in Section 5, then re-run this cell. The "
        "summary is not saved until it is written."
    )

# Per-category reduction as a plain dict; NaN (pest) becomes null in the JSON.
per_category_reduction_json = {
    cat: (float(v) if pd.notna(v) else None)
    for cat, v in per_category_reduction.items()
}

summary = {
    "comparison_target": test_result["comparison_target"],
    "headline_gap_reduction": float(test_result["observed"]),
    "verdict_excludes_zero": test_result["excludes_zero"],
    "statistical_test": test_result,
    "gap_reduction_by_condition": {k: float(v) for k, v in gap_reduction.items()},
    "per_category_reduction": per_category_reduction_json,
    "interpretation": interpretation.strip(),
    "data_variant": DATA_VARIANT,
}

summary_path = OUTPUT_DIR / "comparison_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Wrote {summary_path}")
print(f"  headline targeted - kitchen-sink: {summary['headline_gap_reduction']:+.3f}")
print(f"  verdict (CI excludes zero):       {summary['verdict_excludes_zero']}")    

That is the analysis done. Your Week-4 deliverable is these two files plus your final presentation and your final paper. The paper is where the comparison becomes an argument: it states the verdict and its size, reads the margin against the page's Prediction, and frames the whole thing per Consideration 5 — augmentation reduced the lab→field gap by the amount you measured, and did not close it. The per-category breakdown and the failure-mode reasoning belong in the discussion, as interpretation grounded in the images.

Week 5 is revision: you will take feedback on the paper and the presentation and tighten them. Submit your final presentation and paper to close out Week 4.